# 📊 Notebook 2: EDA & Risk / Volatility Analysis
**FinTech Stock Market Analysis Project**

This notebook covers:
- Exploratory Data Analysis (EDA) across all FinTech stocks
- Correlation analysis
- Risk metrics: Sharpe Ratio, Value at Risk (VaR), Max Drawdown
- Volatility analysis and comparison
- Portfolio performance simulation

## 1. Normalized Price Performance (Base 100)

In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

TICKERS = ['JPM', 'V', 'MA', 'PYPL', 'XYZ', 'COIN', 'HOOD', 'AFRM']
NAMES   = {'JPM':'JPMorgan','V':'Visa','MA':'Mastercard','PYPL':'PayPal',
            'XYZ':'Block','COIN':'Coinbase','HOOD':'Robinhood','AFRM':'Affirm'}

# Load processed data
data = {t: pd.read_csv(f'../data/{t}_processed.csv', index_col=0, parse_dates=True)
        for t in TICKERS if pd.io.common.file_exists(f'../data/{t}_processed.csv')}

TICKERS = list(data.keys())  # only use tickers that downloaded successfully

closes = pd.read_csv('../data/all_closes.csv', index_col=0, parse_dates=True)

print('✅ Data loaded!')
print(f'Tickers: {TICKERS}')
closes.tail()

✅ Data loaded!
Tickers: ['JPM', 'V', 'MA', 'PYPL', 'XYZ', 'COIN', 'HOOD', 'AFRM']


,JPM,V,MA,PYPL,XYZ,COIN,HOOD,AFRM
Date,,,,,,,,
2024-12-23,231.216553,314.334625,524.248474,86.437866,89.290001,268.149994,37.500000,64.949997
2024-12-24,235.018600,317.733429,530.938416,87.780693,91.080002,279.619995,39.580002,65.900002
2024-12-26,235.823608,317.991028,531.374390,87.671272,91.480003,274.410004,40.380001,66.849998
2024-12-27,233.912888,315.761536,527.459595,86.398079,88.970001,265.709991,39.020000,64.639999
2024-12-30,232.118576,312.442017,520.868774,84.975685,87.480003,255.559998,38.279999,62.630001


In [5]:
normalized = (closes / closes.iloc[0]) * 100

fig = go.Figure()
colors = px.colors.qualitative.Bold

for i, ticker in enumerate(TICKERS):
    fig.add_trace(go.Scatter(
        x=normalized.index, y=normalized[ticker],
        name=f'{ticker} ({NAMES[ticker]})',
        line=dict(width=2, color=colors[i % len(colors)])
    ))

fig.add_hline(y=100, line_dash='dash', line_color='white', opacity=0.4)
fig.update_layout(
    template='plotly_dark', height=500,
    title='📈 Normalized FinTech Stock Performance (Base = 100)',
    yaxis_title='Indexed Price (Start = 100)',
    xaxis_title='Date',
    hovermode='x unified'
)
fig.show()

## 2. Return Distribution Analysis

In [6]:
returns = closes.pct_change().dropna()

fig = make_subplots(rows=2, cols=4,
                     subplot_titles=[f'{t} Daily Returns' for t in TICKERS])

for i, ticker in enumerate(TICKERS):
    row = i // 4 + 1
    col = i % 4 + 1
    fig.add_trace(
        go.Histogram(x=returns[ticker], nbinsx=80,
                      marker_color=colors[i % len(colors)],
                      showlegend=False, name=ticker),
        row=row, col=col
    )

fig.update_layout(template='plotly_dark', height=500,
                   title='📊 Daily Return Distributions')
fig.show()

# Stats table
stats = returns.agg(['mean', 'std', 'skew', 'kurt']).T * 100
stats.columns = ['Mean Return (%)', 'Std Dev (%)', 'Skewness', 'Kurtosis']
stats['Annualized Return (%)'] = stats['Mean Return (%)'] * 252
stats['Annualized Volatility (%)'] = stats['Std Dev (%)'] * np.sqrt(252)
print('\n📋 Return Statistics:')
stats.round(3)


📋 Return Statistics:


,Mean Return (%),Std Dev (%),Skewness,Kurtosis,Annualized Return (%),Annualized Volatility (%)
JPM,0.129,1.509,63.324,733.915,32.504,23.956
V,0.084,1.227,7.468,268.690,21.167,19.478
MA,0.085,1.286,-11.272,298.193,21.428,20.418
PYPL,0.052,2.502,-2.824,362.457,13.218,39.718
XYZ,0.088,3.658,26.057,230.909,22.247,58.064
COIN,0.397,5.971,62.951,185.199,100.149,94.783
HOOD,0.300,3.849,32.223,544.393,75.701,61.099
AFRM,0.376,6.217,79.360,338.300,94.843,98.685


## 3. Correlation Heatmap

In [7]:
corr = returns.corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    text=corr.round(2).values,
    texttemplate='%{text}',
    textfont={'size': 12}
))

fig.update_layout(template='plotly_dark', height=500,
                   title='🔥 Return Correlation Matrix — FinTech Stocks')
fig.show()
print('\n💡 Insight: Highly correlated stocks offer less diversification benefit.')


💡 Insight: Highly correlated stocks offer less diversification benefit.


## 4. Risk Metrics: Sharpe Ratio, VaR, Max Drawdown

In [8]:
RISK_FREE_RATE = 0.05 / 252  # 5% annual, daily
CONFIDENCE     = 0.95

risk_metrics = []

for ticker in TICKERS:
    r = returns[ticker].dropna()

    # Sharpe Ratio
    excess_return  = r.mean() - RISK_FREE_RATE
    sharpe         = (excess_return / r.std()) * np.sqrt(252)

    # Value at Risk (Historical)
    var_95         = np.percentile(r, (1 - CONFIDENCE) * 100)
    cvar_95        = r[r <= var_95].mean()  # Conditional VaR / Expected Shortfall

    # Max Drawdown
    cum_returns    = (1 + r).cumprod()
    rolling_max    = cum_returns.cummax()
    drawdown       = (cum_returns - rolling_max) / rolling_max
    max_drawdown   = drawdown.min()

    # Annualized metrics
    ann_return     = r.mean() * 252 * 100
    ann_vol        = r.std() * np.sqrt(252) * 100

    risk_metrics.append({
        'Ticker': ticker,
        'Ann. Return (%)': round(ann_return, 2),
        'Ann. Volatility (%)': round(ann_vol, 2),
        'Sharpe Ratio': round(sharpe, 3),
        'VaR 95% (daily %)': round(var_95 * 100, 3),
        'CVaR 95% (daily %)': round(cvar_95 * 100, 3),
        'Max Drawdown (%)': round(max_drawdown * 100, 2)
    })

risk_df = pd.DataFrame(risk_metrics).set_index('Ticker')
risk_df.to_csv('../data/risk_metrics.csv')
print('📋 Risk Metrics:')
risk_df

📋 Risk Metrics:


,Ann. Return (%),Ann. Volatility (%),Sharpe Ratio,VaR 95% (daily %),CVaR 95% (daily %),Max Drawdown (%)
Ticker,,,,,,
JPM,32.50,23.96,1.148,-2.102,-3.279,-21.51
V,21.17,19.48,0.830,-1.713,-2.753,-18.19
MA,21.43,20.42,0.805,-1.957,-2.961,-21.91
PYPL,13.22,39.72,0.207,-3.730,-5.353,-50.64
XYZ,22.25,58.06,0.297,-5.655,-7.696,-56.67
COIN,100.15,94.78,1.004,-8.489,-10.782,-66.81
HOOD,75.70,61.10,1.157,-4.955,-8.020,-39.56
AFRM,94.84,98.69,0.910,-8.884,-11.848,-77.67


## 5. Risk vs Return Scatter Plot

In [9]:
fig = go.Figure()

for i, ticker in enumerate(TICKERS):
    row = risk_df.loc[ticker]
    fig.add_trace(go.Scatter(
        x=[row['Ann. Volatility (%)']],
        y=[row['Ann. Return (%)']],
        mode='markers+text',
        marker=dict(size=18, color=colors[i % len(colors)]),
        text=[ticker], textposition='top center',
        name=ticker
    ))

fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.3)
fig.update_layout(
    template='plotly_dark', height=500,
    title='⚖️ Risk vs Return — FinTech Stocks',
    xaxis_title='Annualized Volatility (%)',
    yaxis_title='Annualized Return (%)',
    showlegend=False
)
fig.show()
print('💡 Top-left = best (high return, low risk) | Bottom-right = worst')

💡 Top-left = best (high return, low risk) | Bottom-right = worst


## 6. Rolling Volatility Over Time

In [10]:
rolling_vol = returns.rolling(window=30).std() * np.sqrt(252) * 100

fig = go.Figure()
for i, ticker in enumerate(TICKERS):
    fig.add_trace(go.Scatter(
        x=rolling_vol.index, y=rolling_vol[ticker],
        name=ticker, line=dict(width=1.5, color=colors[i % len(colors)])
    ))

fig.update_layout(
    template='plotly_dark', height=450,
    title='📉 30-Day Rolling Annualized Volatility (%)',
    yaxis_title='Volatility (%)', hovermode='x unified'
)
fig.show()
print('\n✅ EDA & Risk Analysis complete! Proceed to Notebook 03.')


✅ EDA & Risk Analysis complete! Proceed to Notebook 03.
